# Deploy Sample MCP Tools

## Overview

This notebook deploys a set of **sample e-commerce MCP tools** that simulate a real order management platform. The tools cover the full customer journey — browsing inventory, placing orders, sending notifications, processing payments, and managing shipments.

## What you will build

Deploy **9 MCP tools** using Amazon Bedrock AgentCore Gateway + Lambda.

By the end you will have:
- **9 MCP tools** running behind an AgentCore Gateway, each backed by an AWS Lambda function
- A **shared sample database** (DynamoDB) that all tools read from and write to
- End-to-end tests that exercise every deployed tool

For A2A agents, see `00_deploy_sample_a2a_agents.ipynb`.

## Architecture

```
MCP Tools via Gateway (9)
──────────────────────────────────
AgentCore Gateway (MCP / HTTP)
  │
  ├─ Lambda: order-management
  │    order_lookup_tool
  │    order_update_tool
  │    order_cancel_tool
  │
  ├─ Lambda: notification
  │    email_send_tool
  │    email_template_tool
  │    sms_notify_tool
  │
  └─ Lambda: read-services
       payment_status_tool
       inventory_check_tool
       shipping_track_tool

Shared: DynamoDB (6 tables) ─── seeded from sample_db.py
```

## Prerequisites
- AWS account with Bedrock model access enabled (`us-west-2`)
- IAM permissions: Lambda, DynamoDB, IAM, Cognito, Bedrock AgentCore
- Python 3.10+

---
## Tools Overview

| Tool | Protocol | Transport | Purpose |
|---|---|---|---|
| `order_lookup_tool` | MCP | `streamable_http` | Look up order details and list orders by customer |
| `order_update_tool` | MCP | `streamable_http` | Update order status or shipping address |
| `order_cancel_tool` | MCP | `streamable_http` | Cancel an order |
| `email_send_tool` | MCP | `streamable_http` | Send transactional emails to customers |
| `email_template_tool` | MCP | `streamable_http` | Manage reusable email templates |
| `sms_notify_tool` | MCP | `streamable_http` | Send SMS notifications |
| `payment_status_tool` | MCP | `streamable_http` | Look up payment status for an order |
| `inventory_check_tool` | MCP | `streamable_http` | Check available stock for one or more SKUs |
| `shipping_track_tool` | MCP | `streamable_http` | Track shipments and get delivery estimates |

## Step 1: Setup

In [ ]:
!pip install strands-agents "strands-agents[a2a]" bedrock-agentcore bedrock-agentcore-starter-toolkit fastapi uvicorn requests nest_asyncio mcp -q

In [ ]:
import boto3, json, time, os, zipfile, io, sys, pathlib, shutil
from datetime import datetime


# Configuration 
AWS_REGION  =  "us-west-2"

# Set AWS credentials if not using Amazon SageMaker notebook
#AWS_PROFILE = "configured-aws-profile"

MODEL_ID    = "us.anthropic.claude-sonnet-4-20250514-v1:0"
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

session          = boto3.Session(region_name=AWS_REGION)
lambda_client     = session.client("lambda")
iam_client        = session.client("iam")
cognito_client    = session.client("cognito-idp")
agentcore_client  = session.client("bedrock-agentcore-control")
ac_data_client    = session.client("bedrock-agentcore")
ACCOUNT_ID        = session.client("sts").get_caller_identity()["Account"]

print(f"Account     : {ACCOUNT_ID}")
print(f"Region      : {AWS_REGION}")
print(f"Timestamp   : {timestamp}")
print("Clients ready.")

In [ ]:
import decimal, json

ddb        = session.resource("dynamodb", region_name=AWS_REGION)
ddb_client = session.client("dynamodb",   region_name=AWS_REGION)

TABLE_PREFIX = f"tools-{timestamp}-"

# Tables used by MCP tools: orders, customers, payments, inventory, shipments, templates
TABLE_DEFS = [
    ("orders",    "order_id",    None),
    ("customers", "customer_id", None),
    ("payments",  "payment_id",  "order_id"),   # GSI: order_id-index
    ("inventory", "sku",         None),
    ("shipments", "shipment_id", "order_id"),   # GSI: order_id-index
    ("templates", "template_id", None),
]

TABLE_NAMES = {}   # logical → actual table name

for logical, pk, gsi_field in TABLE_DEFS:
    tname = TABLE_PREFIX + logical
    TABLE_NAMES[logical] = tname
    kwargs = dict(
        TableName=tname,
        BillingMode="PAY_PER_REQUEST",
        KeySchema=[{"AttributeName": pk, "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": pk, "AttributeType": "S"}],
    )
    if gsi_field:
        kwargs["AttributeDefinitions"].append({"AttributeName": gsi_field, "AttributeType": "S"})
        kwargs["GlobalSecondaryIndexes"] = [{
            "IndexName": f"{gsi_field}-index",
            "KeySchema": [{"AttributeName": gsi_field, "KeyType": "HASH"}],
            "Projection": {"ProjectionType": "ALL"},
        }]
    try:
        ddb_client.create_table(**kwargs)
        print(f"  Creating: {tname}")
    except ddb_client.exceptions.ResourceInUseException:
        print(f"  Already exists: {tname}")

print("Waiting for tables to become ACTIVE...")
for tname in TABLE_NAMES.values():
    ddb_client.get_waiter("table_exists").wait(TableName=tname)
print("All tables ACTIVE.\n")

# ── Seed from utils/sample_db.py ───────────────────────────────────────────
from utils.sample_db import CUSTOMERS, ORDERS, PAYMENTS, INVENTORY, SHIPMENTS, EMAIL_TEMPLATES

def _to_ddb(obj):
    return json.loads(json.dumps(obj), parse_float=decimal.Decimal)

seed_map = {
    "orders":    ORDERS,
    "customers": CUSTOMERS,
    "payments":  PAYMENTS,
    "inventory": INVENTORY,
    "shipments": SHIPMENTS,
    "templates": EMAIL_TEMPLATES,
}

for logical, data in seed_map.items():
    t = ddb.Table(TABLE_NAMES[logical])
    with t.batch_writer() as batch:
        for item in data.values():
            batch.put_item(Item=_to_ddb(item))
    print(f"  Seeded {len(data):2d} records → {TABLE_NAMES[logical]}")

print("\nSample database ready.")
print("\nTable map:")
for k, v in TABLE_NAMES.items():
    print(f"  {k:15s} → {v}")

---
## Step 2: MCP Tools — Order Management

**3 tools, 5 operations** — all served from one Lambda function, three separate Gateway targets.

| Tool | Operations |
|---|---|
| `order_lookup_tool` | `get_order`, `list_orders` |
| `order_update_tool` | `update_order_status`, `update_shipping_addr` |
| `order_cancel_tool` | `cancel_order` |

In [ ]:
# ── Lambda IAM role ────────────────────────────────────────────────────────
lambda_role_name = f"ToolsLambdaRole-{timestamp}"

lambda_role_arn = iam_client.create_role(
    RoleName=lambda_role_name,
    AssumeRolePolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{"Effect": "Allow",
                        "Principal": {"Service": "lambda.amazonaws.com"},
                        "Action": "sts:AssumeRole"}]
    })
)["Role"]["Arn"]

iam_client.attach_role_policy(
    RoleName=lambda_role_name,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
)
iam_client.put_role_policy(
    RoleName=lambda_role_name,
    PolicyName="DynamoDBToolsPolicy",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Effect":   "Allow",
            "Action":   ["dynamodb:GetItem","dynamodb:PutItem","dynamodb:UpdateItem",
                         "dynamodb:Scan","dynamodb:Query"],
            "Resource": [f"arn:aws:dynamodb:{AWS_REGION}:{ACCOUNT_ID}:table/{TABLE_PREFIX}*"]
        }]
    })
)
print(f"Lambda role: {lambda_role_arn}")
print("Waiting 10s for IAM propagation...")
time.sleep(10)

### 2b. Cognito Auth + AgentCore Gateway

One Cognito User Pool and one Gateway are shared by **all 9 MCP tools**. Each tool gets its own Gateway target pointing to its Lambda.

In [ ]:
# ── Cognito User Pool ──────────────────────────────────────────────────────
user_pool_id = cognito_client.create_user_pool(
    PoolName=f"tools-pool-{timestamp}",
    Policies={"PasswordPolicy": {"MinimumLength": 8, "RequireUppercase": False,
                                   "RequireLowercase": False, "RequireNumbers": False,
                                   "RequireSymbols": False}}
)["UserPool"]["Id"]
print(f"User Pool: {user_pool_id}")

resource_server_id = f"tools-api-{timestamp}"
cognito_client.create_resource_server(
    UserPoolId=user_pool_id, Identifier=resource_server_id,
    Name=f"Tools API {timestamp}",
    Scopes=[{"ScopeName": "read",  "ScopeDescription": "Read access"},
            {"ScopeName": "write", "ScopeDescription": "Write access"}]
)

app_client = cognito_client.create_user_pool_client(
    UserPoolId=user_pool_id, ClientName=f"tools-client-{timestamp}",
    GenerateSecret=True, AllowedOAuthFlows=["client_credentials"],
    AllowedOAuthFlowsUserPoolClient=True,
    AllowedOAuthScopes=[f"{resource_server_id}/read", f"{resource_server_id}/write"]
)["UserPoolClient"]
client_id     = app_client["ClientId"]
client_secret = cognito_client.describe_user_pool_client(
    UserPoolId=user_pool_id, ClientId=client_id
)["UserPoolClient"]["ClientSecret"]

cognito_domain_prefix = f"tools-{timestamp}"
cognito_client.create_user_pool_domain(Domain=cognito_domain_prefix, UserPoolId=user_pool_id)
cognito_domain = f"{cognito_domain_prefix}.auth.{AWS_REGION}.amazoncognito.com"
print(f"Client ID : {client_id}")
print(f"Domain    : {cognito_domain}")

# ── Gateway IAM role ───────────────────────────────────────────────────────
gateway_role_name = f"ToolsGatewayRole-{timestamp}"
gateway_role_arn = iam_client.create_role(
    RoleName=gateway_role_name,
    AssumeRolePolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{"Effect": "Allow",
                        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                        "Action": "sts:AssumeRole"}]
    })
)["Role"]["Arn"]
iam_client.attach_role_policy(
    RoleName=gateway_role_name,
    PolicyArn="arn:aws:iam::aws:policy/BedrockAgentCoreFullAccess"
)
iam_client.put_role_policy(
    RoleName=gateway_role_name, PolicyName="LambdaInvokePolicy",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{"Effect": "Allow", "Action": "lambda:InvokeFunction",
                        "Resource": f"arn:aws:lambda:{AWS_REGION}:{ACCOUNT_ID}:function:*-mcp-{timestamp}"}]
    })
)
print(f"Gateway role: {gateway_role_arn}")
time.sleep(10)

# ── Create AgentCore Gateway ───────────────────────────────────────────────
gateway_name = f"tools-mcp-gateway-{timestamp}"
gateway_response = agentcore_client.create_gateway(
    name=gateway_name, roleArn=gateway_role_arn, protocolType="MCP",
    protocolConfiguration={"mcp": {"supportedVersions": ["2025-03-26", "2025-06-18"]}},
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration",
            "allowedClients": [client_id],
            "allowedScopes": [f"{resource_server_id}/read", f"{resource_server_id}/write"]
        }
    }
)
gateway_id = gateway_response["gatewayId"]
print(f"Gateway ID: {gateway_id}")

while True:
    resp = agentcore_client.get_gateway(gatewayIdentifier=gateway_id)
    if resp["status"] == "READY":
        gateway_url = resp["gatewayUrl"]
        print(f"Gateway URL: {gateway_url}")
        break
    print(f"  status: {resp['status']}...")
    time.sleep(10)

In [ ]:
# ── Deploy MCP tool groups (order mgmt, notification, read-services) ───────
# Runs after gateway_id and gateway_role_arn are available.
import importlib
import utils.mcp.order_management, utils.mcp.notification, utils.mcp.read_services
importlib.reload(utils.mcp.order_management)
importlib.reload(utils.mcp.notification)
importlib.reload(utils.mcp.read_services)

from utils.mcp.order_management import deploy as _deploy_order_mgmt
from utils.mcp.notification     import deploy as _deploy_notification
from utils.mcp.read_services    import deploy as _deploy_read_services

_common = dict(
    lambda_client=lambda_client,
    agentcore_client=agentcore_client,
    lambda_role_arn=lambda_role_arn,
    gateway_id=gateway_id,
    gateway_role_arn=gateway_role_arn,
    table_names=TABLE_NAMES,
    timestamp=timestamp,
)

print("=" * 60)
_r = _deploy_order_mgmt(**_common)
order_fn_name          = _r["lambda_fn_name"]
order_lambda_arn       = _r["lambda_arn"]
order_target_id        = _r["targets"]["order_lookup"]
order_update_target_id = _r["targets"]["order_update"]
order_cancel_target_id = _r["targets"]["order_cancel"]

print("=" * 60)
_r = _deploy_notification(**_common)
email_fn_name            = _r["lambda_fn_name"]
email_lambda_arn         = _r["lambda_arn"]
email_target_id          = _r["targets"]["email_send"]
email_template_target_id = _r["targets"]["email_template"]
sms_notify_target_id     = _r["targets"]["sms_notify"]

print("=" * 60)
_r = _deploy_read_services(**_common)
read_fn_name              = _r["lambda_fn_name"]
read_lambda_arn           = _r["lambda_arn"]
payment_status_target_id  = _r["targets"]["payment_status"]
inventory_check_target_id = _r["targets"]["inventory_check"]
shipping_track_target_id  = _r["targets"]["shipping_track"]

print("=" * 60)
print("All MCP tool groups deployed.")

In [ ]:
import base64, requests

# ── Cognito OAuth2 token (re-run if tests return 401 — expires after 1h) ──
credentials = base64.b64encode(f"{client_id}:{client_secret}".encode()).decode()
token_resp  = requests.post(
    f"https://{cognito_domain}/oauth2/token",
    headers={"Authorization": f"Basic {credentials}",
             "Content-Type": "application/x-www-form-urlencoded"},
    data={"grant_type": "client_credentials",
          "scope": f"{resource_server_id}/read {resource_server_id}/write"},
    timeout=30
)
token_resp.raise_for_status()
access_token = token_resp.json()["access_token"]


def _mcp_session():
    h = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}
    r = requests.post(gateway_url, headers=h, json={
        "jsonrpc": "2.0", "id": 0, "method": "initialize",
        "params": {"protocolVersion": "2025-03-26", "capabilities": {},
                   "clientInfo": {"name": "notebook-client", "version": "1.0"}}
    }, timeout=30)
    r.raise_for_status()
    sid = r.headers.get("Mcp-Session-Id")
    if sid:
        h["Mcp-Session-Id"] = sid
    requests.post(gateway_url, headers=h, json={
        "jsonrpc": "2.0", "method": "notifications/initialized", "params": {}
    }, timeout=10)
    return h


def _parse_mcp_result(r):
    ct = r.headers.get("content-type", "")
    if "text/event-stream" in ct:
        for line in r.text.splitlines():
            if line.startswith("data: "):
                data = json.loads(line[6:])
                if "result" in data:
                    content = data["result"].get("content", [])
                    if content and content[0].get("type") == "text":
                        try:    return json.loads(content[0]["text"])
                        except: return content[0]["text"]
                return data
        return {}
    data = r.json()
    if "result" in data:
        content = data["result"].get("content", [])
        if content and content[0].get("type") == "text":
            try:    return json.loads(content[0]["text"])
            except: return content[0]["text"]
    return data


_tool_name_cache = {}

def _resolve_tool_name(h, short_name):
    global _tool_name_cache
    if short_name in _tool_name_cache:
        return _tool_name_cache[short_name]
    r = requests.post(gateway_url, headers=h, json={
        "jsonrpc": "2.0", "id": 99, "method": "tools/list", "params": {}
    }, timeout=30)
    r.raise_for_status()
    tools = r.json().get("result", {}).get("tools", [])
    for t in tools:
        suffix = t["name"].split("___", 1)[-1]
        _tool_name_cache[suffix] = t["name"]
    if short_name not in _tool_name_cache:
        raise ValueError(f"Tool '{short_name}' not found. Available: {[t['name'] for t in tools]}")
    return _tool_name_cache[short_name]


def mcp_call(tool_name, arguments):
    h = _mcp_session()
    full_name = _resolve_tool_name(h, tool_name)
    r = requests.post(gateway_url, headers=h, json={
        "jsonrpc": "2.0", "id": 1, "method": "tools/call",
        "params": {"name": full_name, "arguments": arguments}
    }, timeout=30)
    r.raise_for_status()
    return _parse_mcp_result(r)


def list_mcp_tools():
    h = _mcp_session()
    r = requests.post(gateway_url, headers=h, json={
        "jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}
    }, timeout=30)
    r.raise_for_status()
    return r.json()


print("Helpers ready: mcp_call, list_mcp_tools.")

In [ ]:
# ── Test: Order Management ───────────────────────────────────────────────
print("=" * 60); print("order_lookup_tool — get_order"); print("=" * 60)
print(json.dumps(mcp_call("get_order", {"order_id": "ORD-1002"}), indent=2))

print("=" * 60); print("order_lookup_tool — list_orders"); print("=" * 60)
print(json.dumps(mcp_call("list_orders", {"customer_email": "jane@example.com"}), indent=2))

print("=" * 60); print("order_update_tool — update_order_status"); print("=" * 60)
print(json.dumps(mcp_call("update_order_status", {"order_id": "ORD-1001", "status": "SHIPPED"}), indent=2))

print("=" * 60); print("order_update_tool — update_shipping_addr"); print("=" * 60)
print(json.dumps(mcp_call("update_shipping_addr",
      {"order_id": "ORD-1004", "street": "999 New St", "city": "Denver",
       "state": "CO", "zip": "80201"}), indent=2))

print("=" * 60); print("order_cancel_tool — cancel_order"); print("=" * 60)
print(json.dumps(mcp_call("cancel_order", {"order_id": "ORD-1004", "reason": "customer_request"}), indent=2))


---
## Step 3: MCP Tools — Email and SMS Notifications

**3 tools, 7 operations** — all served from one Lambda function, three separate Gateway targets on the same shared Gateway.

| Tool | Operations |
|---|---|
| `email_send_tool` | `send_email`, `send_bulk_email` |
| `email_template_tool` | `get_template`, `list_templates`, `create_template` |
| `sms_notify_tool` | `send_sms`, `send_bulk_sms` |

In [ ]:
# ── Test: Notifications ──────────────────────────────────────────────────
print("=" * 60); print("email_send_tool — send_email"); print("=" * 60)
print(json.dumps(mcp_call("send_email", {
    "to": "bob@example.com", "template_id": "order_shipped",
    "template_vars": {"customer_name": "Bob Jones", "order_id": "ORD-1002",
                      "tracking_number": "1Z999AA10123456784"}
}), indent=2))

print("=" * 60); print("email_send_tool — send_bulk_email"); print("=" * 60)
print(json.dumps(mcp_call("send_bulk_email", {
    "recipients": ["jane@example.com", "alice@example.com"],
    "subject": "Maintenance notice", "body": "Scheduled maintenance Sunday 2-4am PST."
}), indent=2))

print("=" * 60); print("email_template_tool — list_templates"); print("=" * 60)
print(json.dumps(mcp_call("list_templates", {}), indent=2))

print("=" * 60); print("sms_notify_tool — send_sms"); print("=" * 60)
print(json.dumps(mcp_call("send_sms", {"to": "+15550001002", "body": "Your order ORD-1002 has shipped!"}), indent=2))


---
## Step 4: MCP Tools — Read-Only Services

**3 tools, 5 operations** — all served from one Lambda function, three separate Gateway targets.

| Tool | Operations |
|---|---|
| `payment_status_tool` | `get_payment_status` |
| `inventory_check_tool` | `check_inventory`, `check_multiple_inventory` |
| `shipping_track_tool` | `track_shipment`, `estimate_delivery` |

In [ ]:
# ── Test: Read-Only Services ─────────────────────────────────────────────
print("=" * 60); print("payment_status_tool — get_payment_status"); print("=" * 60)
print(json.dumps(mcp_call("get_payment_status", {"order_id": "ORD-1001"}), indent=2))

print("=" * 60); print("inventory_check_tool — check_multiple_inventory"); print("=" * 60)
print(json.dumps(mcp_call("check_multiple_inventory",
      {"skus": ["WIDGET-42", "GIZMO-3", "GADGET-7"]}), indent=2))

print("=" * 60); print("shipping_track_tool — track_shipment"); print("=" * 60)
print(json.dumps(mcp_call("track_shipment", {"order_id": "ORD-1002"}), indent=2))

print("=" * 60); print("shipping_track_tool — estimate_delivery"); print("=" * 60)
print(json.dumps(mcp_call("estimate_delivery", {"order_id": "ORD-1001"}), indent=2))


---
## Step 5: Summary — All 9 MCP Tools Deployed

### MCP Tools (9) — AgentCore Gateway + Lambda — `streamable_http`

| Step | Tool | Lambda | Operations |
|---|---|---|---|
| 2 | `order_lookup_tool` | order-management | get_order, list_orders |
| 2 | `order_update_tool` | order-management | update_order_status, update_shipping_addr |
| 2 | `order_cancel_tool` | order-management | cancel_order |
| 3 | `email_send_tool` | notification | send_email, send_bulk_email |
| 3 | `email_template_tool` | notification | get_template, list_templates, create_template |
| 3 | `sms_notify_tool` | notification | send_sms, send_bulk_sms |
| 4 | `payment_status_tool` | read-services | get_payment_status |
| 4 | `inventory_check_tool` | read-services | check_inventory, check_multiple_inventory |
| 4 | `shipping_track_tool` | read-services | track_shipment, estimate_delivery |

### Sample Database — DynamoDB (6 tables)

| Table | Seeded records |
|---|---|
| orders | 5 (PROCESSING, SHIPPED, DELIVERED, PENDING, CANCELLED) |
| customers | 4 |
| payments | 5 (CAPTURED ×3, PENDING ×1, REFUNDED ×1) |
| inventory | 4 SKUs (one out-of-stock) |
| shipments | 2 |
| templates | 3 email templates |

For A2A agents, see `00_deploy_sample_a2a_agents.ipynb`.

### Save deployment config for downstream notebooks

Writes `util/mcp_tools_config.json` so that `07_planner_executor_live.ipynb` can load
the Gateway URL, Cognito credentials, and tool definitions without manual copy-paste.

In [ ]:
import pathlib

# Fetch full tool definitions (with inputSchema) from the Gateway
gateway_tools_response = list_mcp_tools()
all_gateway_tools = gateway_tools_response.get("result", {}).get("tools", [])
print(f"Fetched {len(all_gateway_tools)} tools from Gateway")

# Map gateway tools (target___toolname) back to our 9 tool groups
TOOL_GROUP_DEFS = [
    {"name": "order_lookup_tool",    "description": "Retrieve full order details by order ID including status, items, and customer info",
     "tool_names": ["get_order", "list_orders"]},
    {"name": "order_update_tool",    "description": "Update order status, shipping address, or line items for an existing order",
     "tool_names": ["update_order_status", "update_shipping_addr"]},
    {"name": "order_cancel_tool",    "description": "Cancel an order and trigger refund workflow if payment was already captured",
     "tool_names": ["cancel_order"]},
    {"name": "email_send_tool",      "description": "Send transactional emails to customers including order confirmations and shipping updates",
     "tool_names": ["send_email", "send_bulk_email"]},
    {"name": "email_template_tool",  "description": "Manage and render email templates for order confirmations, shipping alerts, and promotions",
     "tool_names": ["list_templates", "render_template"]},
    {"name": "sms_notify_tool",      "description": "Send SMS notifications to customers for time-sensitive order and delivery alerts",
     "tool_names": ["send_sms"]},
    {"name": "payment_status_tool",  "description": "Check payment status, retrieve transaction history, and verify charge details for an order",
     "tool_names": ["get_payment", "list_transactions"]},
    {"name": "inventory_check_tool", "description": "Check real-time stock levels for products across warehouses",
     "tool_names": ["check_stock", "list_warehouses"]},
    {"name": "shipping_track_tool",  "description": "Track shipment status and location in real-time using carrier tracking numbers",
     "tool_names": ["track_shipment", "get_eta"]},
]

# Build a lookup: short_name → full tool definition from Gateway
gateway_tool_lookup = {}
for gt in all_gateway_tools:
    short = gt["name"].split("___", 1)[-1] if "___" in gt["name"] else gt["name"]
    gateway_tool_lookup[short] = gt

# Enrich each tool group with full tool definitions (including inputSchema)
tools_with_schemas = []
for group in TOOL_GROUP_DEFS:
    full_tools = []
    for tn in group["tool_names"]:
        if tn in gateway_tool_lookup:
            gt = gateway_tool_lookup[tn]
            full_tools.append({
                "name": tn,
                "description": gt.get("description", tn),
                "inputSchema": gt.get("inputSchema", {"type": "object"})
            })
        else:
            full_tools.append({"name": tn, "description": tn, "inputSchema": {"type": "object"}})
    tools_with_schemas.append({
        "name": group["name"],
        "description": group["description"],
        "tool_names": group["tool_names"],
        "tools_full": full_tools
    })

mcp_config = {
    "timestamp": timestamp,
    "gateway_id": gateway_id,
    "gateway_url": gateway_url,
    "cognito_domain": cognito_domain,
    "client_id": client_id,
    "client_secret": client_secret,
    "resource_server_id": resource_server_id,
    "user_pool_id": user_pool_id,
    "targets": {
        "order_lookup": order_target_id,
        "order_update": order_update_target_id,
        "order_cancel": order_cancel_target_id,
        "email_send": email_target_id,
        "email_template": email_template_target_id,
        "sms_notify": sms_notify_target_id,
        "payment_status": payment_status_target_id,
        "inventory_check": inventory_check_target_id,
        "shipping_track": shipping_track_target_id,
    },
    "tools": tools_with_schemas
}

config_path = pathlib.Path("utils/mcp_tools_config.json")
config_path.write_text(json.dumps(mcp_config, indent=2))
print(f"Saved: {config_path}")

---
## Step 6: Cleanup (Optional) - PLEASE DO NOT EXECUTE THIS CELL UNTILL YOU HAVE EXECUTED PLANNER EXECUTOR NOTEBOOK

In [ ]:
# ── Cleanup: delete all MCP resources for a given timestamp ────────────────
# Set CLEANUP_TIMESTAMP if different from the current session's `timestamp`.
CLEANUP_TIMESTAMP = ""   # leave empty to use current session's `timestamp`

_ts = CLEANUP_TIMESTAMP or timestamp
print(f"Cleaning up deployment: {_ts}\n")

# ── 1. Gateway targets + gateway ────────────────────────────────────────────
print("Deleting Gateway targets + gateway...")
try:
    _gws = agentcore_client.list_gateways()["items"]
    _gw  = next((g for g in _gws if _ts in g.get("name", "")), None)
    if _gw:
        _gw_id = _gw["gatewayId"]
        try:
            _targets = agentcore_client.list_gateway_targets(gatewayIdentifier=_gw_id)["items"]
            for _t in _targets:
                try:
                    agentcore_client.delete_gateway_target(gatewayIdentifier=_gw_id, targetId=_t["targetId"])
                    print(f"  Deleted target: {_t['name']}")
                except Exception as e:
                    print(f"  {_t['targetId']}: {e}")
        except Exception as e:
            print(f"  list targets: {e}")
        time.sleep(5)
        try:
            agentcore_client.delete_gateway(gatewayIdentifier=_gw_id)
            print(f"  Deleted gateway: {_gw['name']}")
        except Exception as e:
            print(f"  delete gateway: {e}")
    else:
        print("  No matching gateway found")
except Exception as e:
    print(f"  list_gateways failed: {e}")

# ── 2. Lambda functions ──────────────────────────────────────────────────────
print("\nDeleting Lambda functions...")
for _fn in [f"order-mgmt-mcp-{_ts}", f"notification-mcp-{_ts}", f"read-services-mcp-{_ts}"]:
    try:
        lambda_client.delete_function(FunctionName=_fn)
        print(f"  Deleted: {_fn}")
    except lambda_client.exceptions.ResourceNotFoundException:
        print(f"  Not found: {_fn}")
    except Exception as e:
        print(f"  {_fn}: {e}")

# ── 3. Cognito ──────────────────────────────────────────────────────────────
print("\nDeleting Cognito...")
try:
    _pools = cognito_client.list_user_pools(MaxResults=60)["UserPools"]
    _pool  = next((p for p in _pools if p["Name"] == f"tools-pool-{_ts}"), None)
    if _pool:
        _up_id = _pool["Id"]
        try:
            cognito_client.delete_user_pool_domain(Domain=f"tools-{_ts}", UserPoolId=_up_id)
        except Exception:
            pass
        cognito_client.delete_user_pool(UserPoolId=_up_id)
        print(f"  Deleted user pool: {_up_id}")
    else:
        print("  No matching user pool found")
except Exception as e:
    print(f"  {e}")

# ── 4. IAM roles ────────────────────────────────────────────────────────────
print("\nDeleting IAM roles...")

def _delete_role(role_name, inline_policies, managed_arns):
    try:
        for p in inline_policies:
            try:
                iam_client.delete_role_policy(RoleName=role_name, PolicyName=p)
            except Exception:
                pass
        for arn in managed_arns:
            try:
                iam_client.detach_role_policy(RoleName=role_name, PolicyArn=arn)
            except Exception:
                pass
        try:
            for p in iam_client.list_attached_role_policies(RoleName=role_name)["AttachedPolicies"]:
                iam_client.detach_role_policy(RoleName=role_name, PolicyArn=p["PolicyArn"])
        except Exception:
            pass
        try:
            for p in iam_client.list_role_policies(RoleName=role_name)["PolicyNames"]:
                iam_client.delete_role_policy(RoleName=role_name, PolicyName=p)
        except Exception:
            pass
        iam_client.delete_role(RoleName=role_name)
        print(f"  Deleted: {role_name}")
    except iam_client.exceptions.NoSuchEntityException:
        print(f"  Not found: {role_name}")
    except Exception as e:
        print(f"  {role_name}: {e}")

_delete_role(f"ToolsLambdaRole-{_ts}",
             ["DynamoDBToolsPolicy"],
             ["arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"])
_delete_role(f"ToolsGatewayRole-{_ts}",
             ["LambdaInvokePolicy"],
             ["arn:aws:iam::aws:policy/BedrockAgentCoreFullAccess"])

# ── 5. DynamoDB tables ──────────────────────────────────────────────────────
print("\nDeleting DynamoDB tables...")
_prefix = f"tools-{_ts}-"
try:
    _tables = ddb_client.list_tables()["TableNames"]
    for _tname in _tables:
        if _tname.startswith(_prefix):
            try:
                ddb_client.delete_table(TableName=_tname)
                print(f"  Deleted: {_tname}")
            except Exception as e:
                print(f"  {_tname}: {e}")
except Exception as e:
    print(f"  {e}")

# ── 6. Local temp files ──────────────────────────────────────────────────────
print("\nRemoving local temp files...")
for _f in ["db.py", ".bedrock_agentcore.yaml", "Dockerfile", ".dockerignore"]:
    _p = pathlib.Path(_f)
    if _p.exists():
        _p.unlink()
        print(f"  Removed: {_f}")

# ── 7. Config file ────────────────────────────────────────────────────────────
_cfg = pathlib.Path("utils/mcp_tools_config.json")
if _cfg.exists():
    _cfg.unlink()
    print(f"  Removed: {_cfg}")

print("\nCleanup complete.")